In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
.appName('End to End Pipeline with GCS')\
.config('spark.sql.shuffle.partitions','4')\
.getOrCreate()

26/02/07 08:51:47 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
!hadoop fs -ls /user/pipeline_henry/raw_data

Found 9 items
-rw-r--r--   2 prasetyahenry450 hadoop    9033957 2026-02-06 10:15 /user/pipeline_henry/raw_data/olist_customers_dataset.csv
-rw-r--r--   2 prasetyahenry450 hadoop   61273883 2026-02-06 10:15 /user/pipeline_henry/raw_data/olist_geolocation_dataset.csv
-rw-r--r--   2 prasetyahenry450 hadoop   15438671 2026-02-06 10:15 /user/pipeline_henry/raw_data/olist_order_items_dataset.csv
-rw-r--r--   2 prasetyahenry450 hadoop    5777138 2026-02-06 10:15 /user/pipeline_henry/raw_data/olist_order_payments_dataset.csv
-rw-r--r--   2 prasetyahenry450 hadoop   14451670 2026-02-06 10:15 /user/pipeline_henry/raw_data/olist_order_reviews_dataset.csv
-rw-r--r--   2 prasetyahenry450 hadoop   17654914 2026-02-06 10:15 /user/pipeline_henry/raw_data/olist_orders_dataset.csv
-rw-r--r--   2 prasetyahenry450 hadoop    2379446 2026-02-06 10:15 /user/pipeline_henry/raw_data/olist_products_dataset.csv
-rw-r--r--   2 prasetyahenry450 hadoop     174703 2026-02-06 10:15 /user/pipeline_henry/raw_data/olist

In [3]:
# data directory

raw_data = '/user/pipeline_henry/raw_data/'

In [4]:
# Read all raw data files

customers_df = spark.read.csv(raw_data + 'olist_customers_dataset.csv',header=True,inferSchema=True)
orders_df = spark.read.csv(raw_data + 'olist_orders_dataset.csv',header=True,inferSchema=True)
orders_item_df = spark.read.csv(raw_data + 'olist_order_items_dataset.csv',header=True,inferSchema=True)
payments_df = spark.read.csv(raw_data + 'olist_order_payments_dataset.csv',header=True,inferSchema=True)
reviews_df = spark.read.csv(raw_data + 'olist_order_reviews_dataset.csv',header=True,inferSchema=True)
products_df = spark.read.csv(raw_data + 'olist_products_dataset.csv',header=True,inferSchema=True)
sellers_df = spark.read.csv(raw_data + 'olist_sellers_dataset.csv',header=True,inferSchema=True)
geolocation_df = spark.read.csv(raw_data + 'olist_geolocation_dataset.csv',header=True,inferSchema=True)
category_translation_df = spark.read.csv(raw_data + 'product_category_name_translation.csv',header=True,inferSchema=True)

In [5]:
from pyspark.sql.functions import *

In [6]:
# Missing values function
def missing_values(df,df_name):
    print(f'Missing Values in {df_name}:')
    df.select([count(when(col(c).isNull(),1)).alias(c) for c in df.columns]).show()

In [7]:
missing_values(customers_df,'customer')
missing_values(orders_df,'order')
missing_values(orders_item_df,'order_item')
missing_values(payments_df,'payment')
missing_values(reviews_df,'review')
missing_values(products_df,'product')
missing_values(sellers_df,'seller')
missing_values(geolocation_df,'geolocation')
missing_values(category_translation_df,'category')

Missing Values in customer:


+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+

Missing Values in order:


+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+

Missing Values in order_item:
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_i

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        1|    2236|        2380|               92157|                 63079|                8764|                   8785|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+

Missing Values in product:
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---

+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+

Missing Values in category:
+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



## DATA CLEANING

In [8]:
# Drop duplicate values

def clean_dataframe(df,name):
    print("Cleaning "+name)
    return df.dropDuplicates().na.drop('all')

orders_df = clean_dataframe(orders_df,"Orders")

Cleaning Orders


In [9]:
# Data clean orders_df by filling the missing values

orders_df_cleaned = orders_df.fillna({'order_delivered_customer_date':'9999-12-31'})

In [10]:
missing_values(orders_df_cleaned,'order')

Missing Values in order:


+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                            0|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [11]:
payments_df.show()

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [12]:
# Standarize the format from boleto to Bank Transfer

payments_df_cleaned = payments_df.withColumn('payment_type',when(col('payment_type')=='boleto','Bank Transfer')
                                                     .when(col('payment_type')=='credit_card','Credit Card')
                                                     .when(col('payment_type')=='debit_card','Debit Card')
                                                     .otherwise('other'))

In [13]:
payments_df_cleaned.show()

+--------------------+------------------+-------------+--------------------+-------------+
|            order_id|payment_sequential| payment_type|payment_installments|payment_value|
+--------------------+------------------+-------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1|  Credit Card|                   8|        99.33|
|a9810da82917af2d9...|                 1|  Credit Card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1|  Credit Card|                   1|        65.71|
|ba78997921bbcdc13...|                 1|  Credit Card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1|  Credit Card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1|  Credit Card|                   2|        96.12|
|771ee386b001f0620...|                 1|  Credit Card|                   1|        81.16|
|3d7239c394a212faa...|                 1|  Credit Card|                   3|        51.84|

In [14]:
# Change zipcode in customer_df datatype from integer to string

In [15]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [16]:
customers_df_cleaned = customers_df.withColumn('customer_zip_code_prefix',col('customer_zip_code_prefix').cast('string'))

In [17]:
customers_df_cleaned.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [18]:
# Drop missing values score and review_id on reviews_df
reviews_df_cleaned = reviews_df.na.drop(subset=['review_score','review_id'])

In [19]:
reviews_df_cleaned.show()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|                  NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|                NULL|                  NULL| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|                NULL|                  NULL| 2018-02-17 00:00:00|    2018-02-18 14:36:24|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|                NULL|  Recebi bem antes ...| 2017-04-21 00:00:00|   

In [20]:
missing_values(reviews_df_cleaned,'review')

Missing Values in review:
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        0|       0|           0|               89778|                 60699|                6384|                   6404|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



## TRANSFORMATION

In [21]:
# Joining all the dataframes

In [22]:
orders_customers_df = orders_df_cleaned.join(customers_df_cleaned,'customer_id','left')
orders_payments_df = orders_customers_df.join(payments_df_cleaned,'order_id','inner')
orders_items_df = orders_payments_df.join(orders_item_df,'order_id','inner')
orders_items_products_df = orders_items_df.join(products_df, "product_id","left")
orders_seller_df = orders_items_products_df.join(sellers_df, "seller_id","left")
final_df = orders_seller_df.join(reviews_df_cleaned, "order_id","left")

In [23]:
final_df = final_df.join(broadcast(geolocation_df),final_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,'left')

In [24]:
final_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = false)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable 

In [25]:
# Optimize dataframe load using cache

final_df.cache()

26/02/07 08:52:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, seller_id: string, product_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, customer_unique_id: string, customer_zip_code_prefix: string, customer_city: string, customer_state: string, payment_sequential: int, payment_type: string, payment_installments: int, payment_value: double, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, review_id: string, review_score: string, review_comment_title: string, review_comment_message: string, review_creation_da

In [26]:
# Categorizing base on product weight

weight_categorize_df = final_df.withColumn(
    'product_size_category',
    when(col('product_weight_g') <500,'small')
    .when(col('product_weight_g').between(500,2000),'medium')
    .otherwise('large')
)

In [27]:
weight_categorize_df.filter("product_size_category = 'large'").select('product_category_name','product_weight_g','product_size_category').show(5)

+---------------------+----------------+---------------------+
|product_category_name|product_weight_g|product_size_category|
+---------------------+----------------+---------------------+
|                bebes|           10950|                large|
|                bebes|           10950|                large|
|                bebes|           10950|                large|
|                bebes|           10950|                large|
|                bebes|           10950|                large|
+---------------------+----------------+---------------------+
only showing top 5 rows



In [28]:
# Calculate Delivery and Delay on delivery

final_df = final_df.withColumn("actual_delivery_time", datediff("order_delivered_customer_date","order_purchase_timestamp"))
final_df = final_df.withColumn("estimated_delivery_time", datediff("order_estimated_delivery_date","order_purchase_timestamp"))
final_df = final_df.withColumn("delay", col("actual_delivery_time") > col("estimated_delivery_time"))
final_df = final_df.withColumn("delay time", (col("actual_delivery_time") - col("estimated_delivery_time")))

In [29]:
final_df.select('order_status','actual_delivery_time','estimated_delivery_time','delay','delay time').show()

+------------+--------------------+-----------------------+-----+----------+
|order_status|actual_delivery_time|estimated_delivery_time|delay|delay time|
+------------+--------------------+-----------------------+-----+----------+
|   delivered|                   9|                     27|false|       -18|
|   delivered|                   9|                     27|false|       -18|
|   delivered|                   9|                     27|false|       -18|
|   delivered|                   9|                     27|false|       -18|
|   delivered|                   9|                     27|false|       -18|
|   delivered|                   9|                     27|false|       -18|
|   delivered|                   9|                     27|false|       -18|
|   delivered|                   9|                     27|false|       -18|
|   delivered|                   9|                     27|false|       -18|
|   delivered|                   9|                     27|false|       -18|

In [30]:
# Order revenue calculation

final_df = final_df.withColumn('order_revenue',col('price')+col('freight_value'))

In [31]:
final_df.select('price','freight_value','order_revenue').show()

+-----+-------------+-------------+
|price|freight_value|order_revenue|
+-----+-------------+-------------+
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
|159.9|        19.22|       179.12|
+-----+-------------+-------------+
only showing top 20 rows



In [32]:
final_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = false)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable 

# Aggregation for data analytics and dataframe check

In [34]:
# Total order value per order id
order_with_total_value = final_df.groupBy('order_id')\
.agg(sum('payment_value').alias('total_order_value'))

In [35]:
order_with_total_value.show(5)

+--------------------+------------------+
|            order_id| total_order_value|
+--------------------+------------------+
|ce8157b7dce9a36c6...|            168.88|
|4d0a6e1b497dd61e1...| 8683.559999999998|
|fea5a7b9dcb6be7d7...|151578.23999999926|
|2d1f07a3af3df1ddc...|17959.050000000054|
|e58f710ed41fb6825...|10854.059999999996|
+--------------------+------------------+
only showing top 5 rows



In [37]:
# Revenue per seller id
seller_revenue_df = final_df.groupBy('seller_id').agg(sum('price'))

In [38]:
seller_revenue_df.show(5)

+--------------------+--------------------+
|           seller_id|          sum(price)|
+--------------------+--------------------+
|c157bdeedcbc9a8e3...|  105417.29000000014|
|5dceca129747e92ff...|1.4910548339998627E7|
|c70c1b0d8ca86052f...|   6083812.150000138|
|3d01d1c414c44b594...|   45358.49999999993|
|aac29b1b99776be73...|  1326853.8400000301|
+--------------------+--------------------+
only showing top 5 rows



In [39]:
# Total orders per customer
total_orders_customers = final_df.groupBy('customer_id').agg(sum('order_item_id'))

In [40]:
total_orders_customers.show(5)

+--------------------+------------------+
|         customer_id|sum(order_item_id)|
+--------------------+------------------+
|f54a9f0e6b351c431...|                 3|
|52142aa69d8d0e124...|                89|
|569cf68214806a39a...|               215|
|29cb486c739f9774c...|                69|
|e8a332c3433fbd379...|               188|
+--------------------+------------------+
only showing top 5 rows



In [41]:
# Average review score per seller
average_reviews_seller = final_df.groupBy('seller_id').agg(avg('review_score'))

In [42]:
average_reviews_seller.show(5)

+--------------------+------------------+
|           seller_id| avg(review_score)|
+--------------------+------------------+
|633ecdf879b94b533...|1.3549132947976879|
|5dceca129747e92ff...| 4.169185203094777|
|6860153b69cc696d5...| 4.252399544493248|
|16090f2ca825584b5...| 4.132600997601188|
|a6fe7de3d16f6149f...|3.7711071391020923|
+--------------------+------------------+
only showing top 5 rows



In [44]:
# Total product has been sold

most_sold_product_df = final_df.groupBy('product_id').count()

In [45]:
most_sold_product_df.orderBy('count', ascending=False).show(10)

+--------------------+-----+
|          product_id|count|
+--------------------+-----+
|aca2eb7d00ea1a7b8...|86740|
|422879e10f4668299...|81110|
|99a4788cb24856965...|78775|
|389d119b48cf3043d...|60248|
|d1c427060a0f73f6b...|59274|
|368c6c730842d7801...|58358|
|53759a2ecddad2bb8...|52654|
|53b36df67ebb7c415...|52105|
|154e7e31ebfa09220...|42700|
|3dd2a17168ec895c7...|40787|
+--------------------+-----+
only showing top 10 rows



In [46]:
# Total spending per customer
top_customers_spending_df = final_df.groupBy('customer_id').agg(sum('price'))

In [50]:
top_customers_spending_df.show(5)

+--------------------+------------------+
|         customer_id|        sum(price)|
+--------------------+------------------+
|9ef432eb625129730...|2159.2799999999997|
|8ab97904e6daea886...|4417.8000000000075|
|19402a48fe860416a...| 2328.300000000005|
|3187789bec9909876...|6913.5299999999625|
|d2b091571da224a1b...|12968.199999999973|
+--------------------+------------------+
only showing top 5 rows



In [ ]:
# monthly Revenue

In [52]:
montly_revenue = final_df.withColumn('month', date_format('order_purchase_timestamp', 'yyyy-MM')) \
.groupBy('month') \
.agg(
    count('order_id').alias('total_orders'),
    sum('price').alias('total_revenue'),
    round(avg('price'),2).alias('avg_order_value'),
    min('price').alias('min_order_value'),
    max('price').alias('max_orderValues')
)\
.orderBy(desc('month'))

In [53]:
montly_revenue.show(10)

+-------+------------+--------------------+---------------+---------------+---------------+
|  month|total_orders|       total_revenue|avg_order_value|min_order_value|max_orderValues|
+-------+------------+--------------------+---------------+---------------+---------------+
|2018-09|          33|              4785.0|          145.0|          145.0|          145.0|
|2018-08|     1112919| 1.353407940896778E8|         121.61|            2.2|        4399.87|
|2018-07|     1086810| 1.389193056596561E8|         127.82|            3.0|         6729.0|
|2018-06|     1123394|1.4066750211968228E8|         125.22|            3.5|         4590.0|
|2018-05|     1237049|1.5685605658966196E8|          126.8|            3.9|         3930.0|
|2018-04|     1265754|1.5885133711964127E8|          125.5|           0.85|        3399.99|
|2018-03|     1316199|1.5457683251960522E8|         117.44|           4.99|        4099.99|
|2018-02|     1236262|1.4106659914961675E8|         114.11|           2.99|     

In [ ]:
# Yearly Revenue

In [54]:
yearly_revenue = final_df.withColumn('year', date_format('order_purchase_timestamp', 'yyyy')) \
.groupBy('year') \
.agg(
    count('order_id').alias('total_orders'),
    sum('price').alias('total_revenue'),
    round(avg('price'),2).alias('avg_order_value'),
    min('price').alias('min_order_value'),
    max('price').alias('max_orderValues')
)\
.orderBy(desc('year'))

In [55]:
yearly_revenue.show(10)

+----+------------+--------------------+---------------+---------------+---------------+
|year|total_orders|       total_revenue|avg_order_value|min_order_value|max_orderValues|
+----+------------+--------------------+---------------+---------------+---------------+
|2018|     9719186|1.1803210943818722E9|         121.44|           0.85|         6729.0|
|2017|     8280691|1.0041411241984689E9|         121.26|            1.2|         6735.0|
|2016|       63349|    8315395.37999884|         131.26|            6.0|         1399.0|
+----+------------+--------------------+---------------+---------------+---------------+



# Data Serving

In [ ]:
# Data serving in GCS
final_df.write.mode('overwrite').parquet('gs://henry_project/data_serving/')

In [ ]:
# Data serving in hdfs
final_df.write.mode('overwrite').parquet('/user/pipeline_henry/processed/')

In [ ]:
# !hadoop fs -ls -h /user/pipeline_henry/processed/

In [56]:
spark.stop()